In [ ]:
%pip install piexif

# 🗂️ Ordner-Berater v3: Einzeldatei-Verarbeitung mit Metadaten

**Workflow:** Scan → Einzeldatei-Analyse (LLM benennt + Stichwörter) → Metadaten in Dateien schreiben → Übriggebliebene markieren

**Änderungen gegenüber v3-Original:**
- **Skills-Architektur:** Prompts ausgelagert nach `ordner_berater_skills/` — editierbar ohne Code-Änderung
- **Profile:** Lehramt / Allgemein / Forschung — per Dropdown wählbar
- **LLM-Client-Reset:** Automatisch alle N Dateien → verhindert KV-Cache-Degradation
- `MAX_DATEIEN` ersetzt durch `LLM_RESET_INTERVALL` (prominentere Stelle für den wichtigeren Parameter)

## 1. Abhängigkeiten installieren

In [ ]:
try:
    import gradio as gr
    import openai
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "gradio", "openai", "python-docx", "python-pptx",
                           "openpyxl", "PyMuPDF", "Pillow", "piexif",
                           "beautifulsoup4", "striprtf", "odfpy", "xlrd"])
    import gradio as gr
    import openai

print("✅ Abhängigkeiten geladen.")

## 2. Konfiguration

In [ ]:
import os
import json
import hashlib
import re
import shutil
import platform
import subprocess
import tempfile
import base64
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Optional

# ════════════════════════════════════════════════════════════════
# Konfiguration — hier anpassen
# ════════════════════════════════════════════════════════════════

LM_STUDIO_URL = "http://localhost:1234/v1"  # LM Studio Standard-Port
MODELL_NAME = "qwen/qwen3.6-27b"                 # Modellname in LM Studio (wird auto-erkannt)
LLM_RESET_INTERVALL = 50                    # Nach N Dateien: LLM-Client neu erstellen
                                             # Verhindert KV-Cache-Probleme bei langen Sessions
MAX_TIEFE_SCAN = 10                         # Maximale Scan-Tiefe
MAX_SCAN_DATEIEN = 4000                     # Sicherheitslimit beim Scannen
MAX_HASH_SIZE_MB = 50                       # Dateien > X MB nicht hashen

# Undo-Log: Alle Aktionen werden hier protokolliert
UNDO_LOG_PFAD = Path.home() / "ordner_berater_undo.json"

# Papierkorb: Gelöschte Dateien werden hierhin verschoben statt gelöscht
PAPIERKORB_PFAD = Path.home() / "ordner_berater_papierkorb"

# Checkpoint für Einzeldatei-Verarbeitung
CHECKPOINT_PFAD = Path.home() / "ordner_berater_checkpoint.json"

# Skills: Ausgelagerte Prompts + Profile
SKILLS_VERZEICHNIS = Path("ordner_berater_skills")  # Relativ zum Notebook oder absolut
PROFIL_NAME = "lehramt"                     # Verfügbar: lehramt, allgemein, forschung

# Bild-Einstellungen
BILD_ENDUNGEN = {'.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp', '.tiff', '.tif', '.ico'}
METADATEN_BILD_ENDUNGEN = {'.jpg', '.jpeg', '.png'}  # Nur diese bekommen Metadaten direkt
MAX_BILD_MB = 10              # Dateien > X MB überspringen
MAX_BILD_PIXEL = 1024         # Max. Breite/Höhe → ~1000-1500 Vision-Tokens pro Bild
BILD_QUALITAET = 85           # JPEG-Qualität (70-95)

print(f"LM Studio: {LM_STUDIO_URL}")
print(f"Modell: {MODELL_NAME}")
print(f"LLM-Reset alle: {LLM_RESET_INTERVALL} Dateien")
print(f"Profil: {PROFIL_NAME}")
print(f"Skills: {SKILLS_VERZEICHNIS}")
print(f"Undo-Log: {UNDO_LOG_PFAD}")
print(f"Papierkorb: {PAPIERKORB_PFAD}")
print(f"Checkpoint: {CHECKPOINT_PFAD}")

## 3. Kompakt-Scanner (für Einzelordner)

In [ ]:
# ════════════════════════════════════════════════════════════════
# Ignorier-Listen
# ════════════════════════════════════════════════════════════════

IGNORIERTE_VERZEICHNISSE = {
    "node_modules", ".git", "__pycache__", ".ipynb_checkpoints",
    ".venv", "venv", "env", ".env", ".tox", ".mypy_cache",
    ".pytest_cache", "dist", "build", ".next", ".nuxt",
}

IGNORIERTE_DATEIEN = {
    ".DS_Store", "Thumbs.db", "desktop.ini", ".gitkeep",
}

GENERISCHE_NAMEN = re.compile(
    r"^(Untitled\d*|Copy of |Kopie von |.*\(\d+\)$|Neues Dokument|Dokument\d*|"
    r"Unbenannt\d*|Mappe\d*|Tabelle\d*|Präsentation\d*)",
    re.IGNORECASE,
)

VERSIONIERUNGS_MUSTER = re.compile(
    r"(_final|_v\d+|_neu|_alt|_aktuell|_Kopie|_copy|_backup|_old|_new|"
    r"_FINAL|_V\d+|_NEU|_ALT)",
)


def _sha256(pfad: Path, max_bytes: int) -> Optional[str]:
    """SHA-256-Hash berechnen, None wenn zu groß oder Fehler."""
    try:
        if pfad.stat().st_size > max_bytes:
            return None
        h = hashlib.sha256()
        with open(pfad, "rb") as f:
            while chunk := f.read(8192):
                h.update(chunk)
        return h.hexdigest()
    except (PermissionError, OSError):
        return None


def _groesse_lesbar(b: int) -> str:
    """Bytes → menschenlesbare Größe."""
    for einheit in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.0f}{einheit}"
        b /= 1024
    return f"{b:.0f}TB"


def ordner_scannen(basis_pfad: str) -> dict:
    """
    Scannt einen Ordner rekursiv und gibt ein kompaktes Ergebnis zurück.
    Limitiert auf MAX_SCAN_DATEIEN — bei Überschreitung Warnung.
    """
    basis = Path(basis_pfad).resolve()
    if not basis.exists():
        return {"fehler": f"Pfad existiert nicht: {basis}"}
    if not basis.is_dir():
        return {"fehler": f"Kein Ordner: {basis}"}

    dateien = []        # Liste aller Dateien
    ordner = []         # Liste aller Unterordner
    hash_map = defaultdict(list)  # SHA → [Pfade]
    max_hash = MAX_HASH_SIZE_MB * 1024 * 1024
    abgeschnitten = False

    for verz, unterverz, dateinamen in os.walk(basis):
        verz_pfad = Path(verz)
        rel_verz = verz_pfad.relative_to(basis)
        tiefe = len(rel_verz.parts)

        # Ignorierte Verzeichnisse überspringen
        unterverz[:] = [
            d for d in unterverz
            if d not in IGNORIERTE_VERZEICHNISSE and not d.startswith(".")
        ]

        if tiefe > MAX_TIEFE_SCAN:
            continue

        # Ordner erfassen
        ordner.append({
            "pfad": str(verz_pfad),
            "relativ": str(rel_verz) if str(rel_verz) != "." else ".",
            "tiefe": tiefe,
            "anzahl_dateien": 0,  # wird unten gezählt
            "anzahl_unterordner": len(unterverz),
        })

        # Dateien erfassen
        for name in sorted(dateinamen):
            if name in IGNORIERTE_DATEIEN:
                continue

            if len(dateien) >= MAX_SCAN_DATEIEN:
                abgeschnitten = True
                break

            datei_pfad = verz_pfad / name
            try:
                stat = datei_pfad.stat()
            except (PermissionError, OSError):
                continue

            sha = _sha256(datei_pfad, max_hash)
            if sha:
                hash_map[sha].append(str(datei_pfad))

            stamm = datei_pfad.stem
            flags = []
            if GENERISCHE_NAMEN.match(stamm):
                flags.append("GENERISCH")
            if VERSIONIERUNGS_MUSTER.search(stamm):
                flags.append("VERSIONIERT")

            dateien.append({
                "pfad": str(datei_pfad),
                "relativ": str(datei_pfad.relative_to(basis)),
                "name": name,
                "erweiterung": datei_pfad.suffix.lower(),
                "groesse": stat.st_size,
                "groesse_lesbar": _groesse_lesbar(stat.st_size),
                "geaendert": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d"),
                "sha256": sha,
                "flags": flags,
            })

        if abgeschnitten:
            break

    # Dateien pro Ordner zählen
    for d in dateien:
        eltern = str(Path(d["pfad"]).parent)
        for o in ordner:
            if o["pfad"] == eltern:
                o["anzahl_dateien"] += 1
                break

    # Duplikate finden
    duplikate = [
        {"sha256": sha[:12], "dateien": pfade}
        for sha, pfade in hash_map.items()
        if len(pfade) > 1
    ]

    # Dateityp-Verteilung
    typ_counter = Counter(d["erweiterung"] for d in dateien)

    # Max Tiefe
    max_tiefe = max((o["tiefe"] for o in ordner), default=0)

    return {
        "basis_pfad": str(basis),
        "anzahl_dateien": len(dateien),
        "anzahl_ordner": len(ordner),
        "max_tiefe": max_tiefe,
        "abgeschnitten": abgeschnitten,
        "dateien": dateien,
        "ordner": ordner,
        "duplikate": duplikate,
        "dateityp_verteilung": dict(typ_counter.most_common()),
    }


def scan_zusammenfassung_md(scan: dict) -> str:
    """Markdown-Zusammenfassung des Scans für die Gradio-Anzeige."""
    if "fehler" in scan:
        return f"⚠️ {scan['fehler']}"

    z = []
    z.append(f"## Scan: `{scan['basis_pfad']}`")
    z.append(f"")
    z.append(f"| | Wert |")
    z.append(f"|:---|:---|")
    z.append(f"| Dateien | {scan['anzahl_dateien']} |")
    z.append(f"| Ordner | {scan['anzahl_ordner']} |")
    z.append(f"| Max. Tiefe | {scan['max_tiefe']} |")
    z.append(f"| Duplikat-Gruppen | {len(scan['duplikate'])} |")
    if scan['abgeschnitten']:
        z.append(f"| ⚠️ | Abgeschnitten bei {MAX_SCAN_DATEIEN} Dateien |")
    z.append(f"")

    # Top-Dateitypen
    z.append("### Dateitypen")
    z.append("")
    for ext, count in list(scan['dateityp_verteilung'].items())[:10]:
        z.append(f"- `{ext}`: {count}")
    z.append("")

    # Ordnerstruktur (kompakt)
    z.append("### Ordnerstruktur")
    z.append("")
    for o in scan['ordner']:
        einrueckung = "  " * o['tiefe']
        name = Path(o['pfad']).name or "."
        info = f"{o['anzahl_dateien']}D"
        if o['anzahl_unterordner'] > 0:
            info += f", {o['anzahl_unterordner']}U"
        tiefe_warn = " ⚠️" if o['tiefe'] > 4 else ""
        z.append(f"{einrueckung}📁 {name}/ ({info}){tiefe_warn}")
    z.append("")

    # Probleme
    flagged = [d for d in scan['dateien'] if d['flags']]
    if flagged:
        z.append(f"### Auffällige Dateien: {len(flagged)}")
        z.append("")
        for d in flagged[:20]:
            z.append(f"- {'|'.join(d['flags'])} → `{d['relativ']}`")
        if len(flagged) > 20:
            z.append(f"- *... +{len(flagged)-20} weitere*")
        z.append("")

    if scan['duplikate']:
        z.append(f"### Duplikate: {len(scan['duplikate'])} Gruppen")
        z.append("")
        for dup in scan['duplikate'][:10]:
            z.append(f"**Identische Dateien:**")
            for p in dup['dateien']:
                z.append(f"```\n{p}\n```")
        z.append("")

    return "\n".join(z)

def bildformate_konvertieren(basis_pfad: str) -> str:
    """Schritt 0: GIF/BMP/WEBP/ICO/TIFF → PNG konvertieren (wie office_konverter)."""
    try:
        from PIL import Image
    except ImportError:
        return "❌ Pillow nicht installiert."

    basis = Path(basis_pfad).resolve()
    konvertier_endungen = {'.gif', '.bmp', '.webp', '.ico', '.tiff', '.tif'}
    log = []
    undo_eintraege = []
    zeitstempel = datetime.now().isoformat(timespec='seconds')

    for datei in basis.rglob('*'):
        if datei.suffix.lower() not in konvertier_endungen:
            continue
        ziel = datei.with_suffix('.png')
        if ziel.exists():
            log.append(f"⏭️ Ziel existiert: {ziel.name}")
            continue
        try:
            img = Image.open(datei)
            img.load()  # Pixel in RAM laden
            if img.mode not in ('RGB', 'RGBA', 'L', 'LA', 'PA'):
                img = img.convert('RGBA')
            img.save(str(ziel), format='PNG')
            img.close()  # Dateihandle freigeben
            datei.unlink()
            log.append(f"✅ {datei.name} → {ziel.name}")
            undo_eintraege.append({
                'zeit': zeitstempel, 'aktion': 'KONVERTIERT',
                'quelle': str(datei), 'ziel': str(ziel),
                'undo': 'MANUELL',
            })
        except Exception as ex:
            log.append(f"❌ {datei.name}: {str(ex)[:60]}")

    if undo_eintraege:
        _undo_log_schreiben(undo_eintraege)

    if not log:
        return "✅ Keine konvertierbaren Bildformate gefunden."
    return '\n'.join(log)

print("Scanner geladen.")

## 3b. Inhaltsextraktion + Bilderzählung

In [ ]:
# ════════════════════════════════════════════════════════════════
# Umfassende Inhaltsextraktion für Dateinamen-Beratung
# Liest Titel, Überschriften, Inhaltsauszüge aus allen lesbaren Typen.
# NEU: Zählt eingebettete Bilder, extrahiert sie on-demand.
# Fehlende Libs → Typ wird übersprungen, kein Crash.
# ════════════════════════════════════════════════════════════════

# ─── Optionale Imports ─────────────────────────────────────
_hat = {}

try:
    from docx import Document as DocxDocument
    _hat['docx'] = True
except ImportError:
    _hat['docx'] = False

try:
    from pptx import Presentation as PptxPresentation
    _hat['pptx'] = True
except ImportError:
    _hat['pptx'] = False

try:
    import fitz  # PyMuPDF
    _hat['pymupdf'] = True
except ImportError:
    _hat['pymupdf'] = False

try:
    from odf.opendocument import load as odf_load
    from odf import text as odf_text, draw as odf_draw
    _hat['odf'] = True
except ImportError:
    _hat['odf'] = False

try:
    import xlrd
    _hat['xlrd'] = True
except ImportError:
    _hat['xlrd'] = False

try:
    import openpyxl
    _hat['openpyxl'] = True
except ImportError:
    _hat['openpyxl'] = False

try:
    from bs4 import BeautifulSoup
    _hat['bs4'] = True
except ImportError:
    _hat['bs4'] = False

try:
    from striprtf.striprtf import rtf_to_text
    _hat['striprtf'] = True
except ImportError:
    _hat['striprtf'] = False

try:
    import piexif
    _hat['piexif'] = True
except ImportError:
    _hat['piexif'] = False

import xml.etree.ElementTree as ET

print("Bibliotheken-Status:")
for lib, ok in sorted(_hat.items()):
    print(f"  {'✅' if ok else '❌'} {lib}")
print()
fehlend = [k for k, v in _hat.items() if not v]
if fehlend:
    pakete = {'docx': 'python-docx', 'pptx': 'python-pptx', 'pymupdf': 'PyMuPDF',
              'odf': 'odfpy', 'xlrd': 'xlrd', 'openpyxl': 'openpyxl',
              'bs4': 'beautifulsoup4', 'striprtf': 'striprtf', 'piexif': 'piexif'}
    install = ' '.join(pakete.get(k, k) for k in fehlend)
    print(f"Fehlende installieren: pip install {install}")


# ─── Extraktionsfunktionen ─────────────────────────────────
# Jede gibt (titel, extrakt) zurück.

def _ex_ipynb(pfad: Path) -> tuple[str, str]:
    nb = json.loads(pfad.read_text(encoding='utf-8'))
    titel = ""
    md_texte = []
    imports = []
    for cell in nb.get('cells', []):
        src = ''.join(cell.get('source', []))
        if cell.get('cell_type') == 'markdown':
            for line in src.split('\n'):
                if line.strip().startswith('#') and not titel:
                    titel = line.strip().lstrip('#').strip()
            md_texte.append(src.strip()[:200])
        elif cell.get('cell_type') == 'code':
            for line in src.split('\n'):
                if line.strip().startswith(('import ', 'from ')):
                    imports.append(line.strip())
    extrakt = ' | '.join(md_texte[:3])
    if imports:
        extrakt += f" | Imports: {', '.join(imports[:5])}"
    return titel[:120], extrakt[:300]


def _ex_csv(pfad: Path) -> tuple[str, str]:
    with open(pfad, 'r', encoding='utf-8', errors='replace') as f:
        kopf = f.readline().strip()
        zeile2 = f.readline().strip()
    n = sum(1 for _ in open(pfad, 'r', encoding='utf-8', errors='replace')) - 1
    return f"{kopf[:60]}", f"Spalten: {kopf} | Zeile 1: {zeile2} | {n} Zeilen"


def _ex_text(pfad: Path) -> tuple[str, str]:
    with open(pfad, 'r', encoding='utf-8', errors='replace') as f:
        zeilen = [line.strip() for _, line in zip(range(10), f) if line.strip()]
    titel = zeilen[0] if zeilen else ""
    for z in zeilen:
        if z.startswith('#'):
            titel = z.lstrip('#').strip()
            break
    return titel[:120], ' | '.join(zeilen[:5])[:300]


def _ex_py(pfad: Path) -> tuple[str, str]:
    with open(pfad, 'r', encoding='utf-8', errors='replace') as f:
        zeilen = [line.rstrip() for _, line in zip(range(30), f)]
    titel = ""
    beschr = []
    imports = []
    in_docstring = False
    for z in zeilen:
        stripped = z.strip()
        if stripped.startswith('\"\"\"') or stripped.startswith("'''"):
            in_docstring = not in_docstring
            inhalt = stripped.strip('"').strip("'").strip()
            if inhalt:
                beschr.append(inhalt)
        elif in_docstring:
            beschr.append(stripped)
        elif stripped.startswith('#') and not titel:
            titel = stripped.lstrip('#').strip()
        elif stripped.startswith(('import ', 'from ')):
            imports.append(stripped)
    if not titel and beschr:
        titel = beschr[0]
    extrakt = ' '.join(beschr[:3]) if beschr else ' | '.join(imports[:5])
    return titel[:120], extrakt[:300]


def _ex_docx(pfad: Path) -> tuple[str, str]:
    if not _hat.get('docx'):
        return "", ""
    doc = DocxDocument(str(pfad))
    titel = ""
    absaetze = []
    for para in doc.paragraphs[:20]:
        text = para.text.strip()
        if not text:
            continue
        sname = para.style.name.lower()
        if ('heading' in sname or 'überschrift' in sname or 'title' in sname or 'titel' in sname) and not titel:
            titel = text
        absaetze.append(text)
    if not titel and absaetze:
        titel = absaetze[0]
    return titel[:120], ' | '.join(absaetze[:5])[:300]


def _ex_pptx(pfad: Path) -> tuple[str, str]:
    if not _hat.get('pptx'):
        return "", ""
    prs = PptxPresentation(str(pfad))
    folientitel = []
    for slide in prs.slides[:5]:
        if slide.shapes.title and slide.shapes.title.text.strip():
            folientitel.append(slide.shapes.title.text.strip())
    titel = folientitel[0] if folientitel else ""
    return titel[:120], ' → '.join(folientitel)[:300]


def _ex_pdf(pfad: Path) -> tuple[str, str]:
    if not _hat.get('pymupdf'):
        return "", ""
    doc = fitz.open(str(pfad))
    text = ""
    for seite in doc[:2]:  # Erste 2 Seiten
        text += seite.get_text()
    doc.close()
    zeilen = [z.strip() for z in text.split('\n') if z.strip()]
    titel = ""
    for z in zeilen[:10]:
        if len(z) > 5 and not z.isdigit():
            titel = z
            break
    return titel[:120], ' | '.join(zeilen[:8])[:300]


def _ex_xlsx(pfad: Path) -> tuple[str, str]:
    if not _hat.get('openpyxl'):
        return "", ""
    wb = openpyxl.load_workbook(str(pfad), read_only=True, data_only=True)
    blaetter = wb.sheetnames[:5]
    ws = wb.active
    spalten = [str(c.value) for c in next(ws.iter_rows(max_row=1)) if c.value]
    wb.close()
    titel = blaetter[0] if blaetter else ""
    return titel[:120], f"Blätter: {', '.join(blaetter)} | Spalten: {', '.join(spalten[:10])}"[:300]


def _ex_xls(pfad: Path) -> tuple[str, str]:
    if not _hat.get('xlrd'):
        return "", ""
    wb = xlrd.open_workbook(str(pfad))
    blaetter = wb.sheet_names()[:5]
    ws = wb.sheet_by_index(0)
    spalten = [str(ws.cell_value(0, c)) for c in range(min(ws.ncols, 10)) if ws.cell_value(0, c)]
    return blaetter[0][:120] if blaetter else "", f"Blätter: {', '.join(blaetter)} | Spalten: {', '.join(spalten)}"[:300]


def _ex_odt(pfad: Path) -> tuple[str, str]:
    if not _hat.get('odf'):
        return "", ""
    doc = odf_load(str(pfad))
    absaetze = []
    for p in doc.getElementsByType(odf_text.P):
        t = ''.join(node.data for node in p.childNodes if hasattr(node, 'data')).strip()
        if t:
            absaetze.append(t)
        if len(absaetze) >= 10:
            break
    titel = absaetze[0] if absaetze else ""
    return titel[:120], ' | '.join(absaetze[:5])[:300]


def _ex_ods(pfad: Path) -> tuple[str, str]:
    if not _hat.get('odf'):
        return "", ""
    from odf import table as odf_table
    doc = odf_load(str(pfad))
    blaetter = [t.getAttribute('name') for t in doc.getElementsByType(odf_table.Table)][:5]
    titel = blaetter[0] if blaetter else ""
    return titel[:120], f"Blätter: {', '.join(blaetter)}"[:300]


def _ex_odp(pfad: Path) -> tuple[str, str]:
    if not _hat.get('odf'):
        return "", ""
    doc = odf_load(str(pfad))
    from odf.draw import Frame, Page
    texte = []
    for p in doc.getElementsByType(odf_text.P):
        t = ''.join(node.data for node in p.childNodes if hasattr(node, 'data')).strip()
        if t and len(t) > 2:
            texte.append(t)
        if len(texte) >= 8:
            break
    titel = texte[0] if texte else ""
    return titel[:120], ' → '.join(texte[:5])[:300]


def _ex_html(pfad: Path) -> tuple[str, str]:
    text = pfad.read_text(encoding='utf-8', errors='replace')[:10000]
    if _hat.get('bs4'):
        soup = BeautifulSoup(text, 'html.parser')
        titel = soup.title.string.strip() if soup.title and soup.title.string else ""
        h1 = soup.find('h1')
        if h1 and not titel:
            titel = h1.get_text().strip()
        body_text = soup.get_text(separator=' ', strip=True)[:300]
        return titel[:120], body_text
    import re as _re
    m = _re.search(r'<title>(.*?)</title>', text, _re.IGNORECASE | _re.DOTALL)
    titel = m.group(1).strip() if m else ""
    return titel[:120], text[:300]


def _ex_svg(pfad: Path) -> tuple[str, str]:
    try:
        tree = ET.parse(str(pfad))
        root = tree.getroot()
        ns = {'svg': 'http://www.w3.org/2000/svg'}
        titel_el = root.find('.//svg:title', ns) or root.find('.//title')
        titel = titel_el.text.strip() if titel_el is not None and titel_el.text else ""
        texte = []
        for t in (root.findall('.//svg:text', ns) + root.findall('.//text'))[:10]:
            txt = ''.join(t.itertext()).strip()
            if txt:
                texte.append(txt)
        return titel[:120], ' | '.join(texte)[:300]
    except Exception:
        return "", ""


def _ex_xml(pfad: Path) -> tuple[str, str]:
    try:
        tree = ET.parse(str(pfad))
        root = tree.getroot()
        tag = root.tag.split('}')[-1] if '}' in root.tag else root.tag
        kinder = [c.tag.split('}')[-1] for c in root[:10]]
        return tag[:120], f"Root: {tag} | Kinder: {', '.join(kinder)}"[:300]
    except Exception:
        return "", ""


def _ex_json_datei(pfad: Path) -> tuple[str, str]:
    text = pfad.read_text(encoding='utf-8', errors='replace')[:5000]
    daten = json.loads(text)
    if isinstance(daten, dict):
        keys = list(daten.keys())[:10]
        return keys[0] if keys else "", f"Keys: {', '.join(keys)}"[:300]
    elif isinstance(daten, list):
        return f"{len(daten)} Einträge", f"Array mit {len(daten)} Einträgen"[:300]
    return "", str(daten)[:300]


def _ex_yaml(pfad: Path) -> tuple[str, str]:
    with open(pfad, 'r', encoding='utf-8', errors='replace') as f:
        zeilen = [line.rstrip() for _, line in zip(range(15), f) if line.strip()]
    return zeilen[0] if zeilen else "", ' | '.join(zeilen[:5])[:300]


def _ex_rtf(pfad: Path) -> tuple[str, str]:
    if not _hat.get('striprtf'):
        return "", ""
    raw = pfad.read_text(encoding='utf-8', errors='replace')[:10000]
    text = rtf_to_text(raw)
    zeilen = [z.strip() for z in text.split('\n') if z.strip()]
    return zeilen[0][:120] if zeilen else "", ' | '.join(zeilen[:5])[:300]


def _ex_tex(pfad: Path) -> tuple[str, str]:
    text = pfad.read_text(encoding='utf-8', errors='replace')[:5000]
    import re as _re
    titel_match = _re.search(r'\\title\{(.+?)\}', text)
    titel = titel_match.group(1) if titel_match else ""
    sections = _re.findall(r'\\(?:section|chapter)\{(.+?)\}', text)[:5]
    return titel[:120], ' → '.join(sections)[:300] if sections else text[:300]


def _ex_eml(pfad: Path) -> tuple[str, str]:
    import email
    with open(pfad, 'r', encoding='utf-8', errors='replace') as f:
        msg = email.message_from_file(f)
    betreff = msg.get('Subject', '')
    von = msg.get('From', '')
    return betreff[:120], f"Von: {von} | Betreff: {betreff}"[:300]


# ─── Dispatch-Tabelle ──────────────────────────────────────
_EXTRAKTOR = {
    '.docx': _ex_docx, '.pptx': _ex_pptx, '.xlsx': _ex_xlsx,
    '.xls': _ex_xls, '.odt': _ex_odt, '.ods': _ex_ods, '.odp': _ex_odp,
    '.pdf': _ex_pdf,
    '.ipynb': _ex_ipynb, '.py': _ex_py,
    '.txt': _ex_text, '.md': _ex_text, '.rst': _ex_text, '.log': _ex_text,
    '.csv': _ex_csv, '.tsv': _ex_csv, '.json': _ex_json_datei,
    '.yaml': _ex_yaml, '.yml': _ex_yaml,
    '.html': _ex_html, '.htm': _ex_html, '.svg': _ex_svg, '.xml': _ex_xml,
    '.rtf': _ex_rtf, '.tex': _ex_tex, '.eml': _ex_eml,
    '.js': _ex_text, '.ts': _ex_text, '.css': _ex_text,
    '.jsx': _ex_text, '.tsx': _ex_text,
    '.c': _ex_text, '.cpp': _ex_text, '.h': _ex_text, '.hpp': _ex_text,
    '.ino': _ex_text, '.r': _ex_text, '.R': _ex_text, '.jl': _ex_text,
    '.sql': _ex_text, '.sh': _ex_text, '.bat': _ex_text, '.ps1': _ex_text,
    '.lua': _ex_text, '.rb': _ex_text, '.php': _ex_text,
    '.ini': _ex_text, '.cfg': _ex_text, '.toml': _ex_text,
}

_NICHT_EXTRAHIERBAR = {
    '.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp', '.ico', '.tiff',
    '.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv',
    '.mp3', '.wav', '.flac', '.ogg', '.aac', '.wma',
    '.pth', '.pkl', '.joblib', '.h5', '.hdf5', '.onnx',
    '.exe', '.dll', '.so', '.dylib', '.zip', '.tar', '.gz', '.7z', '.rar',
}


# ═══════════════════════════════════════════════════════════════
# Eingebettete Bilder: Zählen (leichtgewichtig, beim Scan)
# und Extrahieren (on-demand, bei der Verarbeitung)
# ═══════════════════════════════════════════════════════════════

MIN_BILD_BYTES = 5 * 1024  # Bilder < 5 KB ignorieren (Icons, Bullets, Deko)


def _bilder_zaehlen_pdf(pfad: Path) -> int:
    """Zählt eingebettete Bildobjekte in einem PDF (ohne sie zu lesen)."""
    if not _hat.get('pymupdf'):
        return 0
    try:
        doc = fitz.open(str(pfad))
        anzahl = 0
        for seite in doc:
            for img in seite.get_images():
                xref = img[0]
                # Größe prüfen ohne das Bild zu extrahieren
                try:
                    info = doc.extract_image(xref)
                    if info and len(info.get("image", b"")) >= MIN_BILD_BYTES:
                        anzahl += 1
                except Exception:
                    pass
        doc.close()
        return anzahl
    except Exception:
        return 0


def _bilder_zaehlen_docx(pfad: Path) -> int:
    """Zählt Bilder in word/media/ des DOCX-ZIP-Archivs."""
    try:
        from zipfile import ZipFile
        with ZipFile(str(pfad), 'r') as zf:
            medien = [n for n in zf.namelist()
                      if n.startswith('word/media/')
                      and zf.getinfo(n).file_size >= MIN_BILD_BYTES]
            return len(medien)
    except Exception:
        return 0


def _bilder_zaehlen_pptx(pfad: Path) -> int:
    """Zählt Bilder in ppt/media/ des PPTX-ZIP-Archivs."""
    try:
        from zipfile import ZipFile
        with ZipFile(str(pfad), 'r') as zf:
            medien = [n for n in zf.namelist()
                      if n.startswith('ppt/media/')
                      and zf.getinfo(n).file_size >= MIN_BILD_BYTES]
            return len(medien)
    except Exception:
        return 0


_BILDER_ZAEHLER = {
    '.pdf': _bilder_zaehlen_pdf,
    '.docx': _bilder_zaehlen_docx,
    '.pptx': _bilder_zaehlen_pptx,
}


def bilder_extrahieren_pdf(pfad: Path, max_bilder: int = 3) -> list[bytes]:
    """Extrahiert die größten eingebetteten Bilder aus einem PDF."""
    if not _hat.get('pymupdf'):
        return []
    try:
        doc = fitz.open(str(pfad))
        alle_bilder = []
        seen_xrefs = set()
        for seite in doc:
            for img in seite.get_images():
                xref = img[0]
                if xref in seen_xrefs:
                    continue
                seen_xrefs.add(xref)
                try:
                    info = doc.extract_image(xref)
                    if info and len(info.get("image", b"")) >= MIN_BILD_BYTES:
                        alle_bilder.append(info["image"])
                except Exception:
                    pass
        doc.close()
        # Größte zuerst, dann top N
        alle_bilder.sort(key=len, reverse=True)
        return alle_bilder[:max_bilder]
    except Exception:
        return []


def bilder_extrahieren_docx(pfad: Path, max_bilder: int = 3) -> list[bytes]:
    """Extrahiert die größten Bilder aus word/media/ eines DOCX."""
    try:
        from zipfile import ZipFile
        with ZipFile(str(pfad), 'r') as zf:
            medien = [(n, zf.getinfo(n).file_size) for n in zf.namelist()
                      if n.startswith('word/media/')
                      and zf.getinfo(n).file_size >= MIN_BILD_BYTES]
            # Größte zuerst
            medien.sort(key=lambda x: x[1], reverse=True)
            return [zf.read(name) for name, _ in medien[:max_bilder]]
    except Exception:
        return []


def bilder_extrahieren_pptx(pfad: Path, max_bilder: int = 3) -> list[bytes]:
    """Extrahiert die größten Bilder aus ppt/media/ eines PPTX."""
    try:
        from zipfile import ZipFile
        with ZipFile(str(pfad), 'r') as zf:
            medien = [(n, zf.getinfo(n).file_size) for n in zf.namelist()
                      if n.startswith('ppt/media/')
                      and zf.getinfo(n).file_size >= MIN_BILD_BYTES]
            medien.sort(key=lambda x: x[1], reverse=True)
            return [zf.read(name) for name, _ in medien[:max_bilder]]
    except Exception:
        return []


_BILDER_EXTRAKTOR = {
    '.pdf': bilder_extrahieren_pdf,
    '.docx': bilder_extrahieren_docx,
    '.pptx': bilder_extrahieren_pptx,
}


def pdf_seite_rendern(pfad: Path, seite_nr: int = 0, dpi: int = 150) -> Optional[bytes]:
    """Fallback: Rendert eine PDF-Seite als JPEG-Bytes (für Scan-PDFs)."""
    if not _hat.get('pymupdf'):
        return None
    try:
        doc = fitz.open(str(pfad))
        if seite_nr >= len(doc):
            doc.close()
            return None
        seite = doc[seite_nr]
        pix = seite.get_pixmap(dpi=dpi)
        bild_bytes = pix.tobytes("jpeg")
        doc.close()
        return bild_bytes
    except Exception:
        return None


# ═══════════════════════════════════════════════════════════════
# Haupt-Extraktionsfunktion (erweitert um Bilderzählung)
# ═══════════════════════════════════════════════════════════════

def inhalte_extrahieren(scan: dict, max_dateien: int = 400) -> dict:
    """
    Ergänzt den Scan um Inhaltsextrakte (titel + extrakt)
    und zählt eingebettete Bilder pro Datei.
    """
    stats = {'extrahiert': 0, 'uebersprungen': 0, 'nicht_lesbar': 0,
             'fehler': 0, 'mit_bildern': 0}

    for d in scan.get('dateien', [])[:max_dateien]:
        ext = d.get('erweiterung', '').lower()

        if ext in _NICHT_EXTRAHIERBAR:
            d['extrakt_status'] = 'binär'
            stats['nicht_lesbar'] += 1
            continue

        extraktor = _EXTRAKTOR.get(ext)
        if not extraktor:
            d['extrakt_status'] = 'kein_extraktor'
            stats['uebersprungen'] += 1
            continue

        if d.get('groesse', 0) > 50 * 1024 * 1024:
            d['extrakt_status'] = 'zu_gross'
            stats['uebersprungen'] += 1
            continue

        try:
            titel, extrakt = extraktor(Path(d['pfad']))
            d['titel'] = titel
            d['extrakt'] = extrakt
            d['extrakt_status'] = 'ok' if (titel or extrakt) else 'leer'
            if titel or extrakt:
                stats['extrahiert'] += 1
        except Exception as ex:
            d['extrakt_status'] = f'fehler: {str(ex)[:50]}'
            stats['fehler'] += 1

        # Eingebettete Bilder zählen (leichtgewichtig)
        bild_zaehler = _BILDER_ZAEHLER.get(ext)
        if bild_zaehler:
            try:
                n_bilder = bild_zaehler(Path(d['pfad']))
                d['anzahl_bilder'] = n_bilder
                if n_bilder > 0:
                    stats['mit_bildern'] += 1
            except Exception:
                d['anzahl_bilder'] = 0
        else:
            d['anzahl_bilder'] = 0

    scan['extraktion_stats'] = stats
    return scan


print(f"Inhaltsextraktion geladen: {len(_EXTRAKTOR)} Dateitypen, "
      f"{len(_BILDER_EXTRAKTOR)} mit Bild-Extraktion (pdf, docx, pptx).")

## 4. LLM-Anbindung + Hilfsfunktionen

In [ ]:
# ════════════════════════════════════════════════════════════════
# LLM-Client + Hilfsfunktionen
# ════════════════════════════════════════════════════════════════

def _llm_client_erstellen():
    """Erstellt einen frischen OpenAI-Client für LM Studio."""
    return openai.OpenAI(
        base_url=LM_STUDIO_URL,
        api_key="nicht-benoetigt",
    )

llm_client = _llm_client_erstellen()
_llm_call_zaehler = 0  # Zählt Calls seit letztem Reset


def _llm_client_reset_wenn_noetig():
    """Setzt den LLM-Client nach LLM_RESET_INTERVALL Calls zurück.
    Erzwingt eine frische HTTP-Session und entlastet den LM Studio KV-Cache.
    """
    global llm_client, _llm_call_zaehler
    _llm_call_zaehler += 1
    if _llm_call_zaehler >= LLM_RESET_INTERVALL:
        llm_client = _llm_client_erstellen()
        _llm_call_zaehler = 0
        return True  # Reset durchgeführt
    return False


def llm_verbindung_testen() -> str:
    """Prüft ob LM Studio erreichbar ist."""
    try:
        modelle = llm_client.models.list()
        namen = [m.id for m in modelle.data]
        return f"✅ LM Studio erreichbar. Modelle: {', '.join(namen)}"
    except Exception as ex:
        return f"❌ LM Studio nicht erreichbar: {ex}\n\nBitte LM Studio starten und Modell laden."


def _llm_json_extrahieren(roh: str):
    """Extrahiert JSON aus einer LLM-Antwort (Thinking + Markdown-Fencing tolerant)."""
    # Thinking-Block entfernen
    if '<think>' in roh:
        roh = re.sub(r'<think>.*?</think>', '', roh, flags=re.DOTALL).strip()

    # Markdown-Fencing entfernen
    if '```' in roh:
        match = re.search(r'```(?:json)?\s*\n?(.*?)\n?```', roh, re.DOTALL)
        if match:
            roh = match.group(1).strip()

    try:
        return json.loads(roh)
    except json.JSONDecodeError:
        # Letzte vollständige } retten
        letzte = roh.rfind('}')
        if letzte > 0:
            try:
                return json.loads(roh[:letzte + 1])
            except json.JSONDecodeError:
                pass
    return None


# ════════════════════════════════════════════════════════════════
# Skill-Loader: Prompts + Profile zur Laufzeit laden
# ════════════════════════════════════════════════════════════════

def _skill_laden(unterverzeichnis: str, name: str) -> str:
    """Lädt eine Skill-Datei aus dem Skills-Verzeichnis."""
    pfad = SKILLS_VERZEICHNIS / unterverzeichnis / f"{name}.md"
    if not pfad.exists():
        raise FileNotFoundError(f"Skill nicht gefunden: {pfad}")
    return pfad.read_text(encoding="utf-8")


def _prompt_zusammenbauen(prompt_name: str, profil_name: str = None) -> str:
    """Lädt einen Prompt und setzt das gewählte Profil ein.
    
    Der Platzhalter {PROFIL} im Prompt wird durch den Inhalt
    der Profil-Datei ersetzt.
    """
    if profil_name is None:
        profil_name = PROFIL_NAME
    prompt = _skill_laden("prompts", prompt_name)
    try:
        profil = _skill_laden("profile", profil_name)
    except FileNotFoundError:
        profil = _skill_laden("profile", "allgemein")
    return prompt.replace("{PROFIL}", profil)


def _verfuegbare_profile() -> list[str]:
    """Gibt die Namen aller verfügbaren Profile zurück."""
    profil_pfad = SKILLS_VERZEICHNIS / "profile"
    if not profil_pfad.exists():
        return ["allgemein"]
    return sorted(p.stem for p in profil_pfad.glob("*.md"))


# ════════════════════════════════════════════════════════════════
# Undo-Log
# ════════════════════════════════════════════════════════════════

def _undo_log_schreiben(eintraege: list[dict]):
    """Schreibt Undo-Einträge in die Log-Datei."""
    bestehend = []
    if UNDO_LOG_PFAD.exists():
        try:
            bestehend = json.loads(UNDO_LOG_PFAD.read_text(encoding="utf-8"))
        except Exception:
            pass
    bestehend.extend(eintraege)
    UNDO_LOG_PFAD.write_text(json.dumps(bestehend, ensure_ascii=False, indent=2), encoding="utf-8")


# ════════════════════════════════════════════════════════════════
# Checkpoint-System
# ════════════════════════════════════════════════════════════════

def _checkpoint_laden() -> dict:
    """Lädt den Checkpoint oder gibt leeres Dict zurück."""
    if CHECKPOINT_PFAD.exists():
        try:
            return json.loads(CHECKPOINT_PFAD.read_text(encoding='utf-8'))
        except Exception:
            pass
    return {"dateien": {}}


def _checkpoint_speichern(checkpoint: dict):
    """Speichert Checkpoint-Datei."""
    CHECKPOINT_PFAD.write_text(
        json.dumps(checkpoint, ensure_ascii=False, indent=2),
        encoding='utf-8'
    )


def _checkpoint_zuruecksetzen():
    """Löscht die Checkpoint-Datei."""
    if CHECKPOINT_PFAD.exists():
        CHECKPOINT_PFAD.unlink()


print("LLM-Anbindung + Checkpoint geladen.")
print(llm_verbindung_testen())

## 5. Metadaten-Schreiber

Schreibt Stichwörter direkt in die Dateien:
- **Office** (.docx, .pptx, .xlsx): `core_properties.keywords`
- **PDF**: PyMuPDF `set_metadata()` — Keywords + Subject
- **JPEG**: EXIF `ImageDescription` via piexif — ohne Neukompression
- **PNG**: `tEXt`-Chunks via Pillow — verlustfrei

In [ ]:
# ════════════════════════════════════════════════════════════════
# Metadaten in Dateien schreiben
# ════════════════════════════════════════════════════════════════
#
# Schreibt Stichwörter direkt in Dateien:
# - Office (.docx, .pptx, .xlsx): core_properties.keywords
# - PDF: PyMuPDF set_metadata() → Keywords + Subject
# - JPEG: EXIF ImageDescription via piexif (keine Neukompression!)
# - PNG: tEXt-Chunks via Pillow PngInfo (verlustfrei)
# ════════════════════════════════════════════════════════════════


def _stichworte_als_string(stichworte: list[str]) -> str:
    """Liste → kommaseparierter String für Metadaten-Felder."""
    return ", ".join(s.strip() for s in stichworte if s.strip())


def metadaten_schreiben_docx(pfad: Path, stichworte: list[str]) -> str:
    """Stichwörter in .docx core_properties.keywords schreiben."""
    if not _hat.get('docx'):
        return "❌ python-docx nicht installiert"
    try:
        doc = DocxDocument(str(pfad))
        kw_str = _stichworte_als_string(stichworte)
        # Bestehende Keywords ergänzen, nicht überschreiben
        bestehend = doc.core_properties.keywords or ""
        if bestehend:
            alle = set(k.strip() for k in bestehend.split(",") if k.strip())
            alle.update(stichworte)
            kw_str = _stichworte_als_string(sorted(alle))
        doc.core_properties.keywords = kw_str
        doc.save(str(pfad))
        return f"✅ docx Keywords: {kw_str[:80]}"
    except Exception as ex:
        return f"❌ docx Fehler: {str(ex)[:80]}"


def metadaten_schreiben_pptx(pfad: Path, stichworte: list[str]) -> str:
    """Stichwörter in .pptx core_properties.keywords schreiben."""
    if not _hat.get('pptx'):
        return "❌ python-pptx nicht installiert"
    try:
        prs = PptxPresentation(str(pfad))
        kw_str = _stichworte_als_string(stichworte)
        bestehend = prs.core_properties.keywords or ""
        if bestehend:
            alle = set(k.strip() for k in bestehend.split(",") if k.strip())
            alle.update(stichworte)
            kw_str = _stichworte_als_string(sorted(alle))
        prs.core_properties.keywords = kw_str
        prs.save(str(pfad))
        return f"✅ pptx Keywords: {kw_str[:80]}"
    except Exception as ex:
        return f"❌ pptx Fehler: {str(ex)[:80]}"


def metadaten_schreiben_xlsx(pfad: Path, stichworte: list[str]) -> str:
    """Stichwörter in .xlsx properties.keywords schreiben."""
    if not _hat.get('openpyxl'):
        return "❌ openpyxl nicht installiert"
    try:
        wb = openpyxl.load_workbook(str(pfad))
        kw_str = _stichworte_als_string(stichworte)
        bestehend = wb.properties.keywords or ""
        if bestehend:
            alle = set(k.strip() for k in bestehend.split(",") if k.strip())
            alle.update(stichworte)
            kw_str = _stichworte_als_string(sorted(alle))
        wb.properties.keywords = kw_str
        wb.save(str(pfad))
        return f"✅ xlsx Keywords: {kw_str[:80]}"
    except Exception as ex:
        return f"❌ xlsx Fehler: {str(ex)[:80]}"


def metadaten_schreiben_pdf(pfad: Path, stichworte: list[str]) -> str:
    """Stichwörter in PDF-Metadata schreiben (PyMuPDF, inkrementell)."""
    if not _hat.get('pymupdf'):
        return "❌ PyMuPDF nicht installiert"
    try:
        doc = fitz.open(str(pfad))
        kw_str = _stichworte_als_string(stichworte)
        # Bestehende Metadata lesen und ergänzen
        meta = doc.metadata or {}
        bestehend = meta.get("keywords", "") or ""
        if bestehend:
            alle = set(k.strip() for k in bestehend.split(",") if k.strip())
            alle.update(stichworte)
            kw_str = _stichworte_als_string(sorted(alle))
        doc.set_metadata({
            "keywords": kw_str,
            "subject": kw_str[:120],  # Subject als Kurz-Zusammenfassung
        })
        doc.saveIncr()  # Inkrementelles Speichern — schnell, keine vollständige Neukodierung
        doc.close()
        return f"✅ pdf Keywords: {kw_str[:80]}"
    except Exception as ex:
        return f"❌ pdf Fehler: {str(ex)[:80]}"


def metadaten_schreiben_jpeg(pfad: Path, stichworte: list[str], beschreibung: str = "") -> str:
    """EXIF-Metadaten in JPEG schreiben via Pillow (quality=98, praktisch verlustfrei)."""
    try:
        from PIL import Image

        kw_str = _stichworte_als_string(stichworte)
        beschr = beschreibung or kw_str

        img = Image.open(pfad)
        img.load()

        # Bestehende EXIF beibehalten und ergänzen
        exif = img.getexif()
        exif[270] = beschr       # ImageDescription
        exif[40092] = kw_str.encode('utf-16le')  # XPKeywords (Windows-Explorer suchbar!)

        img.save(str(pfad), quality='keep', exif=exif.tobytes(), subsampling='keep')
        img.close()
        return f"✅ jpeg EXIF: {kw_str[:80]}"
    except Exception as ex:
        return f"❌ jpeg Fehler: {str(ex)[:80]}"


def metadaten_schreiben_png(pfad: Path, stichworte: list[str], beschreibung: str = "") -> str:
    """PNG tEXt-Chunks schreiben — verlustfrei, Datei wird neu gespeichert."""
    try:
        from PIL import Image
        from PIL.PngImagePlugin import PngInfo

        kw_str = _stichworte_als_string(stichworte)
        beschr = beschreibung or kw_str

        img = Image.open(pfad)

        # Bestehende Metadaten beibehalten
        metadata = PngInfo()
        if hasattr(img, 'text') and img.text:
            for key, val in img.text.items():
                if key not in ('Keywords', 'Description'):
                    metadata.add_text(key, str(val))

        metadata.add_text("Keywords", kw_str)
        metadata.add_text("Description", beschr)

        img.save(str(pfad), pnginfo=metadata)
        return f"✅ png tEXt: {kw_str[:80]}"
    except Exception as ex:
        return f"❌ png Fehler: {str(ex)[:80]}"


# ─── Dispatch ──────────────────────────────────────────────

_METADATEN_SCHREIBER = {
    '.docx': metadaten_schreiben_docx,
    '.pptx': metadaten_schreiben_pptx,
    '.xlsx': metadaten_schreiben_xlsx,
    '.pdf':  metadaten_schreiben_pdf,
    '.jpg':  metadaten_schreiben_jpeg,
    '.jpeg': metadaten_schreiben_jpeg,
    '.png':  metadaten_schreiben_png,
}

# Bild-Schreiber brauchen extra "beschreibung"-Parameter
_BILD_METADATEN_ENDUNGEN = {'.jpg', '.jpeg', '.png'}


def metadaten_schreiben(pfad: Path, stichworte: list[str], beschreibung: str = "") -> str:
    """Universeller Dispatcher: schreibt Metadaten wenn Format unterstützt."""
    ext = pfad.suffix.lower()
    schreiber = _METADATEN_SCHREIBER.get(ext)
    if not schreiber:
        return f"⏭️ Kein Metadaten-Support für {ext}"

    if ext in _BILD_METADATEN_ENDUNGEN:
        return schreiber(pfad, stichworte, beschreibung)
    else:
        return schreiber(pfad, stichworte)


print(f"Metadaten-Schreiber geladen: {', '.join(sorted(_METADATEN_SCHREIBER.keys()))}")

## 6. Einzeldatei-Verarbeitung (Drei-Kanal-Architektur)

Drei Verarbeitungspfade pro Datei:
1. **Text-only** → Dateien ohne eingebettete Bilder
2. **Hybrid** (Text + Bild) → Text-LLM bekommt Extrakt UND eingebettete Bilder
3. **Vision-only** → Standalone-Bilder, Scan-PDFs (Seite 1 rendern als Fallback)

**Prompts werden aus `ordner_berater_skills/` geladen** (nicht mehr hartcodiert).
Profile (lehramt, allgemein, forschung) bestimmen Fachkürzel und Typ-Präfixe.

**LLM-Client wird alle `LLM_RESET_INTERVALL` Dateien zurückgesetzt**
um KV-Cache-Probleme bei langen Sessions zu vermeiden.

In [ ]:
# ════════════════════════════════════════════════════════════════
# Prompts für Einzeldatei-Verarbeitung
# ════════════════════════════════════════════════════════════════

# ════════════════════════════════════════════════════════════════
# Prompts aus Skills laden (statt hartcodiert)
# ════════════════════════════════════════════════════════════════

def _prompts_laden(profil_name: str = None) -> tuple[str, str, str]:
    """Lädt alle drei Prompts mit dem gewählten Profil.
    Wird bei jedem Verarbeitungslauf aufgerufen, damit Profil-Wechsel
    im Gradio-Interface sofort wirksam werden.
    """
    text_prompt = _prompt_zusammenbauen("text_benennung", profil_name)
    hybrid_prompt = _prompt_zusammenbauen("hybrid_benennung", profil_name)
    vision_prompt = _prompt_zusammenbauen("vision_benennung", profil_name)
    return text_prompt, hybrid_prompt, vision_prompt


# Initiales Laden (wird in schritt2_verarbeiten ggf. mit anderem Profil überschrieben)
try:
    TEXT_EINZEL_PROMPT, HYBRID_EINZEL_PROMPT, VISION_EINZEL_PROMPT = _prompts_laden()
    print(f"✅ Prompts geladen (Profil: {PROFIL_NAME})")
    print(f"   Verfügbare Profile: {_verfuegbare_profile()}")
except FileNotFoundError as e:
    print(f"⚠️ Skill-Dateien nicht gefunden: {e}")
    print(f"   Erwartet in: {SKILLS_VERZEICHNIS.resolve()}")
    print(f"   → Prompts werden beim Verarbeitungsstart geprüft.")
    TEXT_EINZEL_PROMPT = HYBRID_EINZEL_PROMPT = VISION_EINZEL_PROMPT = ""


# ════════════════════════════════════════════════════════════════
# Bild-Encoding Hilfsfunktionen
# ════════════════════════════════════════════════════════════════

def _bild_zu_base64(pfad: Path) -> Optional[str]:
    """Liest Bilddatei von Disk, skaliert, gibt base64-Data-URI zurück."""
    try:
        groesse_mb = pfad.stat().st_size / (1024 * 1024)
        if groesse_mb > MAX_BILD_MB:
            return None

        from PIL import Image
        import io

        img = Image.open(pfad)

        if img.mode in ('RGBA', 'PA', 'LA'):
            hintergrund = Image.new('RGB', img.size, (255, 255, 255))
            hintergrund.paste(img, mask=img.split()[-1])
            img = hintergrund
        elif img.mode not in ('RGB', 'L'):
            img = img.convert('RGB')

        w, h = img.size
        if w > MAX_BILD_PIXEL or h > MAX_BILD_PIXEL:
            faktor = MAX_BILD_PIXEL / max(w, h)
            img = img.resize((int(w * faktor), int(h * faktor)), Image.LANCZOS)

        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=BILD_QUALITAET)
        daten = base64.b64encode(buffer.getvalue()).decode('utf-8')
        return f"data:image/jpeg;base64,{daten}"
    except Exception:
        try:
            mime_types = {
                '.png': 'image/png', '.jpg': 'image/jpeg', '.jpeg': 'image/jpeg',
                '.gif': 'image/gif', '.bmp': 'image/bmp', '.webp': 'image/webp',
            }
            mime = mime_types.get(pfad.suffix.lower(), 'image/png')
            with open(pfad, 'rb') as f:
                daten = base64.b64encode(f.read()).decode('utf-8')
            return f"data:{mime};base64,{daten}"
        except Exception:
            return None


def _bild_bytes_zu_base64(bild_bytes: bytes, max_pixel: int = 1024) -> Optional[str]:
    """Konvertiert Bild-Bytes (aus PDF/DOCX extrahiert) zu base64-Data-URI."""
    try:
        from PIL import Image
        import io

        img = Image.open(io.BytesIO(bild_bytes))

        # Alpha entfernen
        if img.mode in ('RGBA', 'PA', 'LA'):
            bg = Image.new('RGB', img.size, (255, 255, 255))
            bg.paste(img, mask=img.split()[-1])
            img = bg
        elif img.mode not in ('RGB', 'L'):
            img = img.convert('RGB')

        # Skalieren
        w, h = img.size
        if w > max_pixel or h > max_pixel:
            faktor = max_pixel / max(w, h)
            img = img.resize((int(w * faktor), int(h * faktor)), Image.LANCZOS)

        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=BILD_QUALITAET)
        daten = base64.b64encode(buffer.getvalue()).decode('utf-8')
        return f"data:image/jpeg;base64,{daten}"
    except Exception:
        return None


# ════════════════════════════════════════════════════════════════
# Verarbeitungspfade: Text-only, Hybrid (Text+Bild), Vision-only
# ════════════════════════════════════════════════════════════════

def _einzeldatei_text_verarbeiten(datei: dict) -> dict:
    """Kanal 1: Nur Text — für Dateien ohne eingebettete Bilder."""
    pfad = datei['pfad']
    name = datei['name']
    ext = datei['erweiterung']
    titel = datei.get('titel', '')
    extrakt = datei.get('extrakt', '')

    inhalt_teile = [f"Datei: {name} | Typ: {ext}"]
    if titel:
        inhalt_teile.append(f"TITEL: {titel}")
    if extrakt:
        inhalt_teile.append(f"EXTRAKT: {extrakt}")

    try:
        _llm_client_reset_wenn_noetig()
        antwort = llm_client.chat.completions.create(
            model=MODELL_NAME,
            messages=[
                {'role': 'system', 'content': TEXT_EINZEL_PROMPT},
                {'role': 'user', 'content': '\n'.join(inhalt_teile)},
            ],
            temperature=0.2,
            max_tokens=1024,
        )
        roh = antwort.choices[0].message.content.strip()
        ergebnis = _llm_json_extrahieren(roh)

        if isinstance(ergebnis, dict):
            return {
                'pfad': pfad, 'alter_name': name,
                'neuer_name': ergebnis.get('neuer_name', name),
                'grund': ergebnis.get('grund', ''),
                'stichworte': ergebnis.get('stichworte', []),
                'typ': 'text', 'status': 'ok', 'kanal': 'text',
            }
    except Exception as ex:
        return {
            'pfad': pfad, 'alter_name': name, 'neuer_name': name,
            'grund': f'Fehler: {str(ex)[:80]}', 'stichworte': [],
            'typ': 'text', 'status': 'fehler', 'kanal': 'text',
        }

    return {
        'pfad': pfad, 'alter_name': name, 'neuer_name': name,
        'grund': 'JSON-Parsing fehlgeschlagen', 'stichworte': [],
        'typ': 'text', 'status': 'fehler', 'kanal': 'text',
    }


def _einzeldatei_hybrid_verarbeiten(datei: dict, bild_uris: list[str]) -> dict:
    """Kanal 2: Text + eingebettete Bilder in einem Call (Qwen3 multimodal)."""
    pfad = datei['pfad']
    name = datei['name']
    ext = datei['erweiterung']
    titel = datei.get('titel', '')
    extrakt = datei.get('extrakt', '')

    # User-Content zusammenbauen: Text + Bilder
    text_teil = f"Datei: {name} | Typ: {ext}"
    if titel:
        text_teil += f"\nTITEL: {titel}"
    if extrakt:
        text_teil += f"\nEXTRAKT: {extrakt}"
    text_teil += f"\n\nAnzahl eingebettete Bilder: {len(bild_uris)}"

    user_content = [{'type': 'text', 'text': text_teil}]
    for i, uri in enumerate(bild_uris):
        user_content.append({
            'type': 'text',
            'text': f'\nEingebettetes Bild {i+1}/{len(bild_uris)}:'
        })
        user_content.append({
            'type': 'image_url',
            'image_url': {'url': uri}
        })

    try:
        _llm_client_reset_wenn_noetig()
        antwort = llm_client.chat.completions.create(
            model=MODELL_NAME,
            messages=[
                {'role': 'system', 'content': HYBRID_EINZEL_PROMPT},
                {'role': 'user', 'content': user_content},
            ],
            temperature=0.2,
            max_tokens=1024,
        )
        roh = antwort.choices[0].message.content.strip()
        ergebnis = _llm_json_extrahieren(roh)

        if isinstance(ergebnis, dict):
            return {
                'pfad': pfad, 'alter_name': name,
                'neuer_name': ergebnis.get('neuer_name', name),
                'grund': ergebnis.get('grund', ''),
                'stichworte': ergebnis.get('stichworte', []),
                'typ': 'hybrid', 'status': 'ok',
                'kanal': f'hybrid ({len(bild_uris)} Bilder)',
            }
    except Exception as ex:
        # Fallback: nur Text verarbeiten wenn Hybrid fehlschlägt
        return _einzeldatei_text_verarbeiten(datei)

    return _einzeldatei_text_verarbeiten(datei)


def _einzeldatei_bild_verarbeiten(datei: dict) -> dict:
    """Kanal 3a: Vision-only — für standalone Bilddateien."""
    pfad_str = datei['pfad']
    pfad = Path(pfad_str)
    name = datei['name']
    ext = datei['erweiterung']

    data_uri = _bild_zu_base64(pfad)
    if not data_uri:
        return {
            'pfad': pfad_str, 'alter_name': name, 'neuer_name': name,
            'grund': 'Bild zu gross oder nicht lesbar', 'beschreibung': '',
            'stichworte': [], 'konfidenz': 'niedrig',
            'typ': 'bild', 'status': 'fehler', 'kanal': 'vision',
        }

    try:
        _llm_client_reset_wenn_noetig()
        antwort = llm_client.chat.completions.create(
            model=MODELL_NAME,
            messages=[
                {'role': 'system', 'content': VISION_EINZEL_PROMPT},
                {'role': 'user', 'content': [
                    {'type': 'text', 'text': f'Analysiere dieses Bild. Aktuelle Datei: {name} (Endung: {ext})'},
                    {'type': 'image_url', 'image_url': {'url': data_uri}},
                ]},
            ],
            temperature=0.2,
            max_tokens=512,
        )
        roh = antwort.choices[0].message.content.strip()
        ergebnis = _llm_json_extrahieren(roh)

        if isinstance(ergebnis, dict):
            neuer_name = ergebnis.get('neuer_name')
            if neuer_name and not neuer_name.lower().endswith(ext.lower()):
                neuer_name = f"{Path(neuer_name).stem}{ext}"
            return {
                'pfad': pfad_str, 'alter_name': name,
                'neuer_name': neuer_name or name,
                'grund': ergebnis.get('beschreibung', ''),
                'beschreibung': ergebnis.get('beschreibung', ''),
                'stichworte': ergebnis.get('stichworte', []),
                'konfidenz': ergebnis.get('konfidenz', 'niedrig'),
                'typ': 'bild', 'status': 'ok' if neuer_name else 'nicht_erkannt',
                'kanal': 'vision',
            }
    except Exception:
        pass

    return {
        'pfad': pfad_str, 'alter_name': name, 'neuer_name': name,
        'grund': 'Vision-Fehler', 'beschreibung': '', 'stichworte': [],
        'konfidenz': 'niedrig', 'typ': 'bild', 'status': 'fehler',
        'kanal': 'vision',
    }


def _einzeldatei_pdf_vision_fallback(datei: dict) -> dict:
    """Kanal 3b: Scan-PDF Fallback — Seite 1 rendern und an Vision schicken."""
    pfad_str = datei['pfad']
    pfad = Path(pfad_str)
    name = datei['name']
    ext = datei['erweiterung']

    bild_bytes = pdf_seite_rendern(pfad, seite_nr=0, dpi=150)
    if not bild_bytes:
        return {
            'pfad': pfad_str, 'alter_name': name, 'neuer_name': name,
            'grund': 'PDF-Rendering fehlgeschlagen', 'stichworte': [],
            'typ': 'pdf_scan', 'status': 'fehler', 'kanal': 'pdf_vision_fallback',
        }

    data_uri = _bild_bytes_zu_base64(bild_bytes)
    if not data_uri:
        return {
            'pfad': pfad_str, 'alter_name': name, 'neuer_name': name,
            'grund': 'Bild-Encoding fehlgeschlagen', 'stichworte': [],
            'typ': 'pdf_scan', 'status': 'fehler', 'kanal': 'pdf_vision_fallback',
        }

    try:
        _llm_client_reset_wenn_noetig()
        antwort = llm_client.chat.completions.create(
            model=MODELL_NAME,
            messages=[
                {'role': 'system', 'content': VISION_EINZEL_PROMPT},
                {'role': 'user', 'content': [
                    {'type': 'text', 'text': (
                        f'Das ist Seite 1 eines PDF-Dokuments: {name}\n'
                        f'Der Text konnte nicht automatisch extrahiert werden.\n'
                        f'Analysiere das gerenderte Seitenbild und vergib '
                        f'einen aussagekräftigen Dateinamen (Endung: {ext}).'
                    )},
                    {'type': 'image_url', 'image_url': {'url': data_uri}},
                ]},
            ],
            temperature=0.2,
            max_tokens=1024,
        )
        roh = antwort.choices[0].message.content.strip()
        ergebnis = _llm_json_extrahieren(roh)

        if isinstance(ergebnis, dict):
            neuer_name = ergebnis.get('neuer_name')
            if neuer_name and not neuer_name.lower().endswith(ext.lower()):
                neuer_name = f"{Path(neuer_name).stem}{ext}"
            return {
                'pfad': pfad_str, 'alter_name': name,
                'neuer_name': neuer_name or name,
                'grund': ergebnis.get('beschreibung', ergebnis.get('grund', '')),
                'beschreibung': ergebnis.get('beschreibung', ''),
                'stichworte': ergebnis.get('stichworte', []),
                'typ': 'pdf_scan', 'status': 'ok' if neuer_name else 'nicht_erkannt',
                'kanal': 'pdf_vision_fallback',
            }
    except Exception:
        pass

    return {
        'pfad': pfad_str, 'alter_name': name, 'neuer_name': name,
        'grund': 'Vision-Fallback fehlgeschlagen', 'stichworte': [],
        'typ': 'pdf_scan', 'status': 'fehler', 'kanal': 'pdf_vision_fallback',
    }


# ════════════════════════════════════════════════════════════════
# Dispatcher: Wählt den richtigen Verarbeitungspfad
# ════════════════════════════════════════════════════════════════

def einzeldatei_verarbeiten(datei: dict) -> dict:
    """
    Drei-Kanal-Dispatcher:
    1. Text vorhanden + Bilder → Hybrid (Text + Bilder in einem Call)
    2. Text vorhanden, keine Bilder → Text-only
    3. Standalone Bild → Vision-only
    4. PDF ohne Text, ohne extrahierbare Bilder → Seite rendern (Fallback)
    5. Alles andere → überspringen
    """
    ext = datei.get('erweiterung', '').lower()
    hat_text = datei.get('extrakt_status') == 'ok'
    n_bilder = datei.get('anzahl_bilder', 0)
    ist_bild = ext in BILD_ENDUNGEN
    ist_pdf = ext == '.pdf'
    
    # Standalone Bilddateien → Vision
    if ist_bild:
        return _einzeldatei_bild_verarbeiten(datei)

    # Dateien mit Text UND eingebetteten Bildern → Hybrid
    if hat_text and n_bilder > 0:
        bilder_extraktor = _BILDER_EXTRAKTOR.get(ext)
        if bilder_extraktor:
            bild_bytes_liste = bilder_extraktor(Path(datei['pfad']), max_bilder=3)
            bild_uris = []
            for bild_bytes in bild_bytes_liste:
                uri = _bild_bytes_zu_base64(bild_bytes)
                if uri:
                    bild_uris.append(uri)
            if bild_uris:
                return _einzeldatei_hybrid_verarbeiten(datei, bild_uris)
        # Fallback: nur Text wenn Bildextraktion fehlschlägt
        return _einzeldatei_text_verarbeiten(datei)

    # Dateien mit Text, ohne Bilder → Text-only
    if hat_text:
        return _einzeldatei_text_verarbeiten(datei)

    # PDF ohne brauchbaren Text → Entscheidungsbaum
    if ist_pdf:
        # Erst prüfen: hat das PDF extrahierbare Bildobjekte?
        if n_bilder > 0:
            bild_bytes_liste = bilder_extrahieren_pdf(Path(datei['pfad']), max_bilder=3)
            bild_uris = []
            for bild_bytes in bild_bytes_liste:
                uri = _bild_bytes_zu_base64(bild_bytes)
                if uri:
                    bild_uris.append(uri)
            if bild_uris:
                # PDF hat Bilder aber keinen Text → Vision mit extrahierten Bildern
                return _einzeldatei_hybrid_verarbeiten(datei, bild_uris)
        # Letzter Ausweg: Seite 1 rendern
        return _einzeldatei_pdf_vision_fallback(datei)

    # Alles andere: nicht verarbeitbar
    return {
        'pfad': datei['pfad'], 'alter_name': datei['name'],
        'neuer_name': datei['name'], 'grund': 'Kein Extrakt / Binaerdatei',
        'stichworte': [], 'typ': 'uebersprungen', 'status': 'uebersprungen',
        'kanal': 'keine',
    }


# ════════════════════════════════════════════════════════════════
# Ergebnis-Formatierung
# ════════════════════════════════════════════════════════════════

def ergebnisse_als_markdown(ergebnisse: list[dict]) -> str:
    """Formatiert die Verarbeitungsergebnisse als Markdown-Tabelle."""
    if not ergebnisse:
        return "Keine Ergebnisse."

    z = [f"## Verarbeitungsergebnis: {len(ergebnisse)} Dateien\n"]
    z.append("| # | Kanal | Alter Name | Neuer Name | Stichwörter | Metadaten |")
    z.append("|:--|:--|:--|:--|:--|:--|")

    for i, e in enumerate(ergebnisse, 1):
        if e.get('status') == 'uebersprungen' and e.get('typ') == 'uebersprungen':
            continue

        kanal = e.get('kanal', '?')[:12]
        alter = e.get('alter_name', '?')[:25]
        neuer = e.get('neuer_name', '?')[:25]
        kw = ', '.join(e.get('stichworte', [])[:4])[:35]
        meta = e.get('metadaten_status', '')[:15]

        if alter == neuer:
            neuer = "—"
        z.append(f"| {i} | {kanal} | `{alter}` | `{neuer}` | {kw} | {meta} |")

    return '\n'.join(z)


print("Einzeldatei-Verarbeitung geladen (Drei-Kanal: Text / Hybrid / Vision).")

## 7. Übriggebliebene Dateien markieren

Dateien ohne Extrakt/Vision-Ergebnis bekommen ein Präfix:
- `_binaer_` — Modelle, Archive, Binärdateien
- `_bild_fehler_` — Bilder, die nicht erkannt wurden
- `_text_fehler_` — Textdateien mit Extraktionsfehler
- `_manuell_` — Kein Extraktor vorhanden

In [ ]:
# ════════════════════════════════════════════════════════════════
# Markierung der Übriggebliebenen
# ════════════════════════════════════════════════════════════════

def uebriggebliebene_analysieren(scan: dict, ergebnisse: list[dict]) -> tuple[list[dict], str]:
    """
    Identifiziert alle Dateien, die nicht erfolgreich verarbeitet wurden.
    Nutzt die Ergebnisliste der Einzeldatei-Verarbeitung.
    """
    # Pfade der erfolgreich verarbeiteten Dateien
    verarbeitet_pfade = set()
    for e in ergebnisse:
        if e.get('status') == 'verarbeitet' and (e.get('umbenannt') or e.get('stichworte')):
            verarbeitet_pfade.add(e.get('pfad', ''))

    kategorien = {
        'binaer': [],
        'bild_fehler': [],
        'text_fehler': [],
        'manuell': [],
    }

    for d in scan.get('dateien', []):
        pfad_str = d.get('pfad', '')
        name = d.get('name', '')
        ext = d.get('erweiterung', '').lower()
        status = d.get('extrakt_status', '')

        # Bereits verarbeitet?
        if pfad_str in verarbeitet_pfade:
            continue

        # Bereits mit bekanntem Präfix?
        bekannte_praefixe = (
            'ab_', 'ka_', 'ewh_', 'praes_', 'verlauf_', 'handout_',
            'info_', 'lsg_', 'daten_', 'modell_', 'skript_',
            'notebook_', 'tab_', 'dok_', 'bild_',
        )
        if name.lower().startswith(bekannte_praefixe):
            continue

        # Bereits markiert?
        if name.startswith('_'):
            continue

        # Kategorisieren
        if ext in BILD_ENDUNGEN:
            kategorien['bild_fehler'].append(d)
        elif ext in _NICHT_EXTRAHIERBAR:
            kategorien['binaer'].append(d)
        elif status and ('fehler' in status or status == 'leer'):
            kategorien['text_fehler'].append(d)
        elif status == 'kein_extraktor':
            kategorien['manuell'].append(d)
        else:
            kategorien['manuell'].append(d)

    gesamt = sum(len(v) for v in kategorien.values())
    dateien_flat = []
    for kat, dateien_liste in kategorien.items():
        for d in dateien_liste:
            dateien_flat.append({**d, 'markierung_kat': kat})

    kat_info = {
        'binaer': ('🔵', 'Binäre Dateien (.pth, .pkl, .zip, ...)', '_binaer_'),
        'bild_fehler': ('🟠', 'Bilder — nicht erkannt vom Vision-LLM', '_bild_fehler_'),
        'text_fehler': ('🟡', 'Textdateien — Extraktion fehlgeschlagen', '_text_fehler_'),
        'manuell': ('⚪', 'Kein Extraktor vorhanden', '_manuell_'),
    }

    z = [f'## Übriggebliebene: {gesamt} Dateien\n']
    for kat, dateien_liste in kategorien.items():
        icon, beschr, praefix = kat_info[kat]
        if not dateien_liste:
            continue
        z.append(f'### {icon} {beschr}: {len(dateien_liste)} Dateien (Markierung: `{praefix}`)')
        z.append('')

        nach_ordner = defaultdict(list)
        for d in dateien_liste:
            nach_ordner[str(Path(d['pfad']).parent)].append(d)

        for ordner, ord_dateien in sorted(nach_ordner.items()):
            z.append(f'📁 `{ordner}`')
            for d in ord_dateien[:15]:
                z.append(f'- `{d["name"]}` ({d.get("groesse_lesbar", "?")})')
            if len(ord_dateien) > 15:
                z.append(f'- *... +{len(ord_dateien)-15} weitere*')
            z.append('')

    return dateien_flat, '\n'.join(z)


def uebriggebliebene_markieren(dateien: list[dict], genehmigt_labels: list[str]) -> str:
    """Markiert genehmigte Dateien mit Präfix im Dateinamen."""
    if not genehmigt_labels:
        return 'Keine Dateien zur Markierung ausgewählt.'

    genehmigt_nummern = set()
    for label in genehmigt_labels:
        match = re.match(r'#(\d+)', label)
        if match:
            genehmigt_nummern.add(int(match.group(1)))

    kat_praefix = {
        'binaer': '_binaer_',
        'bild_fehler': '_bild_fehler_',
        'text_fehler': '_text_fehler_',
        'manuell': '_manuell_',
    }

    log = []
    undo_eintraege = []
    zeitstempel = datetime.now().isoformat(timespec='seconds')

    for i, d in enumerate(dateien, 1):
        if i not in genehmigt_nummern:
            continue

        pfad = Path(d['pfad'])
        kat = d.get('markierung_kat', 'manuell')
        praefix = kat_praefix.get(kat, '_manuell_')

        if pfad.name.startswith('_'):
            log.append(f'⏭️ #{i} Bereits markiert: `{pfad.name}`')
            continue

        neuer_name = f"{praefix}{pfad.name}"
        ziel = pfad.parent / neuer_name

        if not pfad.exists():
            log.append(f'❌ #{i} Existiert nicht: `{pfad.name}`')
            continue
        if ziel.exists():
            log.append(f'❌ #{i} Ziel existiert: `{neuer_name}`')
            continue

        try:
            pfad.rename(ziel)
            log.append(f'✅ #{i} `{pfad.name}` → `{neuer_name}`')
            undo_eintraege.append({
                'zeit': zeitstempel, 'aktion': 'UMBENANNT',
                'quelle': str(pfad), 'ziel': str(ziel),
                'undo': 'ZURUECK_UMBENENNEN'
            })
        except Exception as ex:
            log.append(f'❌ #{i} Fehler: {ex}')

    if undo_eintraege:
        _undo_log_schreiben(undo_eintraege)

    log.append(f'\n---\n{len(undo_eintraege)} Dateien markiert. Undo-Log: `{UNDO_LOG_PFAD}`')
    return '\n\n'.join(log)


def uebrig_als_auswahl(dateien: list[dict]) -> list[str]:
    """Erstellt Checkbox-Labels für Übriggebliebene."""
    kat_icon = {'binaer': '🔵', 'bild_fehler': '🟠', 'text_fehler': '🟡', 'manuell': '⚪'}
    kat_praefix = {
        'binaer': '_binaer_', 'bild_fehler': '_bild_fehler_',
        'text_fehler': '_text_fehler_', 'manuell': '_manuell_',
    }
    labels = []
    for i, d in enumerate(dateien, 1):
        kat = d.get('markierung_kat', 'manuell')
        icon = kat_icon.get(kat, '⚪')
        pfix = kat_praefix.get(kat, '_manuell_')
        labels.append(f"#{i} {icon} {d['name']} → {pfix}{d['name']}")
    return labels


print('Markierung Übriggebliebener geladen.')

## 8. Gradio-Interface

**Workflow:**
1. Ordner scannen + Inhalte extrahieren + Bilder zählen
2. Einzeldatei-Verarbeitung (Text / Hybrid / Vision → Name + Stichwörter + Metadaten)
3. Übriggebliebene markieren

In [ ]:
# ════════════════════════════════════════════════════════════════
# Globaler Zustand
# ════════════════════════════════════════════════════════════════

_aktueller_scan: dict = {}
_aktuelle_ergebnisse: list[dict] = []
_aktuelle_uebrige: list[dict] = []


# ════════════════════════════════════════════════════════════════
# Schritt 1: Scannen
# ════════════════════════════════════════════════════════════════

def schritt1_scannen(pfad: str):
    """Schritt 1: Ordner scannen + Inhalte extrahieren."""
    global _aktueller_scan, _aktuelle_ergebnisse
    _aktuelle_ergebnisse = []

    if not pfad or not pfad.strip():
        return "⚠️ Bitte einen Pfad eingeben."

    scan = ordner_scannen(pfad.strip())
    if "fehler" in scan:
        return f"⚠️ {scan['fehler']}"

    _aktueller_scan = scan

    # Inhaltsextraktion durchführen
    scan = inhalte_extrahieren(scan)
    _aktueller_scan = scan

    md = scan_zusammenfassung_md(scan)

    # Statistik anzeigen
    stats = scan.get('extraktion_stats', {})
    bild_anzahl = sum(1 for d in scan.get('dateien', [])
                      if d.get('erweiterung', '').lower() in BILD_ENDUNGEN)

    if stats.get('extrahiert', 0) > 0 or bild_anzahl > 0:
        md += (f"\n\n📖 **Verarbeitbar:** "
               f"{stats.get('extrahiert', 0)} Textdateien + "
               f"{bild_anzahl} Bilder")

    # Eingebettete-Bilder-Info
    mit_bildern = stats.get('mit_bildern', 0)
    if mit_bildern > 0:
        gesamt_bilder = sum(d.get('anzahl_bilder', 0) for d in scan.get('dateien', []))
        md += (f"\n🖼️ **Eingebettete Bilder:** {mit_bildern} Dateien enthalten "
               f"insgesamt {gesamt_bilder} Bilder → Hybrid-Verarbeitung (Text+Bild)")

    # Checkpoint-Info
    checkpoint = _checkpoint_laden()
    cp_basis = checkpoint.get('basis_pfad', '')
    scan_basis = str(Path(pfad.strip()).resolve())
    if cp_basis == scan_basis:
        n_cp = len(checkpoint.get('dateien', {}))
        md += f"\n\n💾 **Checkpoint gefunden:** {n_cp} Dateien bereits verarbeitet. Wird fortgesetzt."

    return md


# ════════════════════════════════════════════════════════════════
# Schritt 2: Einzeldatei-Verarbeitung (Generator für Live-Fortschritt)
# ════════════════════════════════════════════════════════════════

def schritt2_verarbeiten(profil_wahl: str = None):
    """
    Einzeldatei-Verarbeitung mit Fortschritts-Updates via Generator.
    Gradio ruft den Generator iterativ auf und zeigt Zwischenstände.
    """
    global _aktuelle_ergebnisse

    if not _aktueller_scan or 'fehler' in _aktueller_scan:
        yield '⚠️ Zuerst einen Ordner scannen.', ''
        return

    # Prompts mit gewähltem Profil laden
    global TEXT_EINZEL_PROMPT, HYBRID_EINZEL_PROMPT, VISION_EINZEL_PROMPT
    global _llm_call_zaehler
    aktives_profil = profil_wahl or PROFIL_NAME
    try:
        TEXT_EINZEL_PROMPT, HYBRID_EINZEL_PROMPT, VISION_EINZEL_PROMPT = _prompts_laden(aktives_profil)
    except FileNotFoundError as e:
        yield f"❌ Skill-Dateien nicht gefunden: {e}", ""
        return
    _llm_call_zaehler = 0  # Reset-Zähler zurücksetzen

    dateien = _aktueller_scan.get('dateien', [])
    kandidaten = [d for d in dateien
                  if d.get('extrakt_status') == 'ok'          # Text vorhanden
                  or d.get('erweiterung', '').lower() in BILD_ENDUNGEN  # Bilddatei
                  or d.get('anzahl_bilder', 0) > 0             # Hat eingebettete Bilder
                  or d.get('erweiterung', '').lower() == '.pdf']  # PDF → Fallback möglich

    if not kandidaten:
        yield '⚠️ Keine verarbeitbaren Dateien gefunden.', ''
        return

    checkpoint = _checkpoint_laden()
    fortschritt_zeilen = [f'🔄 Starte Verarbeitung: {len(kandidaten)} Dateien...']
    yield '\n'.join(fortschritt_zeilen), ''

    ergebnisse = []
    undo_eintraege = []
    zeitstempel = datetime.now().isoformat(timespec='seconds')
    umbenannt = 0
    metadaten_ok = 0
    uebersprungen = 0

    for i, datei in enumerate(kandidaten):
        pfad_str = datei['pfad']

        # Checkpoint-Check: bereits verarbeitet?
        if pfad_str in checkpoint.get('dateien', {}):
            cp = checkpoint['dateien'][pfad_str]
            if cp.get('status') in ('verarbeitet', 'uebersprungen'):
                uebersprungen += 1
                ergebnisse.append(cp)
                fortschritt_zeilen.append(
                    f'⏭️ {i+1}/{len(kandidaten)} {datei["name"]} — Checkpoint')
                # Nur alle 10 Dateien updaten (Gradio-Performance)
                if i % 10 == 0:
                    yield '\n'.join(fortschritt_zeilen[-20:]), ergebnisse_als_markdown(ergebnisse)
                continue

        # Fortschritt anzeigen
        fortschritt_zeilen.append(f'🔄 {i+1}/{len(kandidaten)} {datei["name"]}...')
        yield '\n'.join(fortschritt_zeilen[-20:]), ergebnisse_als_markdown(ergebnisse)

        # LLM verarbeiten (ein Call pro Datei)
        ergebnis = einzeldatei_verarbeiten(datei)
        pfad = Path(pfad_str)

        # Umbenennung durchführen
        neuer_name = ergebnis.get('neuer_name', '')
        umbenennung_ok = False
        if (neuer_name
                and neuer_name != ergebnis.get('alter_name')
                and ergebnis.get('status') == 'ok'):
            ziel = pfad.parent / neuer_name
            # Sicherheitschecks
            if (pfad.exists()
                    and not ziel.exists()
                    and pfad.suffix.lower() == Path(neuer_name).suffix.lower()):
                try:
                    pfad.rename(ziel)
                    umbenennung_ok = True
                    umbenannt += 1
                    undo_eintraege.append({
                        'zeit': zeitstempel, 'aktion': 'UMBENANNT',
                        'quelle': str(pfad), 'ziel': str(ziel),
                        'undo': 'ZURUECK_UMBENENNEN'
                    })
                    pfad = ziel  # Für Metadaten den neuen Pfad verwenden
                except Exception as ex:
                    ergebnis['umbenennung_fehler'] = str(ex)[:80]

        ergebnis['umbenannt'] = umbenennung_ok

        # Metadaten schreiben
        stichworte = ergebnis.get('stichworte', [])
        meta_status = ""
        if stichworte and pfad.exists():
            beschreibung = ergebnis.get('beschreibung', ergebnis.get('grund', ''))
            meta_status = metadaten_schreiben(pfad, stichworte, beschreibung)
            if meta_status.startswith('✅'):
                metadaten_ok += 1

        ergebnis['metadaten_status'] = meta_status
        ergebnis['status'] = 'verarbeitet'
        ergebnisse.append(ergebnis)

        # Checkpoint speichern (nach jeder Datei!)
        checkpoint.setdefault('dateien', {})[pfad_str] = ergebnis
        checkpoint['basis_pfad'] = _aktueller_scan.get('basis_pfad', '')
        checkpoint['letzte_aktualisierung'] = zeitstempel
        _checkpoint_speichern(checkpoint)

        # Status-Zeile aktualisieren
        icon = '✅' if umbenennung_ok else ('🏷️' if stichworte else '⏭️')
        kanal = ergebnis.get('kanal', '?')
        suffix = f' → {neuer_name}' if umbenennung_ok else ''
        # Reset-Info anzeigen wenn gerade zurückgesetzt
        if _llm_call_zaehler == 0 and i > 0:
            fortschritt_zeilen.append(f'🔄 LLM-Client Reset nach {LLM_RESET_INTERVALL} Calls')
        fortschritt_zeilen[-1] = f'{icon} {i+1}/{len(kandidaten)} [{kanal}] {datei["name"]}{suffix}'
        yield '\n'.join(fortschritt_zeilen[-20:]), ergebnisse_als_markdown(ergebnisse)

    # Undo-Log schreiben
    if undo_eintraege:
        _undo_log_schreiben(undo_eintraege)

    _aktuelle_ergebnisse = ergebnisse

    # Abschlussbericht
    abschluss = (
        f"\n---\n"
        f"## ✅ Fertig!\n\n"
        f"| | Anzahl |\n"
        f"|:--|:--|\n"
        f"| Verarbeitet | {len(ergebnisse)} |\n"
        f"| Umbenannt | {umbenannt} |\n"
        f"| Metadaten geschrieben | {metadaten_ok} |\n"
        f"| Aus Checkpoint übersprungen | {uebersprungen} |\n"
        f"\n"
        f"📁 Checkpoint: `{CHECKPOINT_PFAD}`\n"
        f"📜 Undo-Log: `{UNDO_LOG_PFAD}`"
    )
    fortschritt_zeilen.append(abschluss)
    yield '\n'.join(fortschritt_zeilen[-25:]), ergebnisse_als_markdown(ergebnisse)


# ════════════════════════════════════════════════════════════════
# Schritt 3: Übriggebliebene
# ════════════════════════════════════════════════════════════════

def schritt3_uebrige_analysieren():
    """Identifiziert nicht verarbeitete Dateien."""
    global _aktuelle_uebrige
    if not _aktueller_scan or 'fehler' in _aktueller_scan:
        return '⚠️ Zuerst scannen.', gr.update(choices=[], value=[])

    dateien, md_text = uebriggebliebene_analysieren(_aktueller_scan, _aktuelle_ergebnisse)
    _aktuelle_uebrige = dateien
    labels = uebrig_als_auswahl(dateien)
    return md_text, gr.update(choices=labels, value=[])


def schritt3_uebrige_markieren(genehmigt: list[str]):
    """Führt genehmigte Markierungen durch."""
    if not _aktuelle_uebrige:
        return '⚠️ Zuerst Übriggebliebene analysieren.'
    return uebriggebliebene_markieren(_aktuelle_uebrige, genehmigt)


def uebrige_alle_auswaehlen(auswahl: list[str]):
    """Toggle: alle auswählen / alle abwählen."""
    labels = uebrig_als_auswahl(_aktuelle_uebrige)
    if len(auswahl) == len(labels):
        return gr.update(value=[])
    return gr.update(value=labels)


def checkpoint_zuruecksetzen():
    """Löscht die Checkpoint-Datei."""
    _checkpoint_zuruecksetzen()
    return "🗑️ Checkpoint gelöscht. Nächste Verarbeitung startet von vorn."


# ════════════════════════════════════════════════════════════════
# Gradio App
# ════════════════════════════════════════════════════════════════

with gr.Blocks(
    title="Ordner-Berater v3",
    theme=gr.themes.Soft(),
) as app:

    gr.Markdown("# 🗂️ Ordner-Berater v3")
    gr.Markdown(
        "**3-Schritte-Workflow:** Scan → Einzeldatei-Verarbeitung "
        "(Name + Stichwörter + Metadaten) → Übriggebliebene markieren\n\n"
        "**Sicherheit:** Alle Umbenennungen im Undo-Log. "
        "Checkpoint nach jeder Datei."
    )

    # ─── Schritt 1: Scannen ────────────────────────────────
    gr.Markdown("---")
    gr.Markdown("### Schritt 1: Ordner scannen")

    with gr.Row():
        pfad_eingabe = gr.Textbox(
            label="Ordnerpfad",
            placeholder="z.B. D:/Lehramt/Chemie oder C:/Users/name/Documents/Projekt",
            scale=3,
        )
        btn_scan = gr.Button("🔍 Scannen", variant="primary", scale=1)

    scan_ausgabe = gr.Markdown(value="*Pfad eingeben und scannen.*")
    
    # ─── Schritt 0: Bildformate konvertieren ──────────────
    gr.Markdown("---")
    gr.Markdown("### Schritt 0: Bildformate konvertieren (optional)")
    gr.Markdown("Konvertiert GIF/BMP/WEBP/ICO/TIFF → PNG (Metadaten-fähig). "
                "Nur nötig beim ersten Durchlauf.")
    btn_konvertieren = gr.Button("🔄 Bildformate → PNG", variant="secondary")
    konvert_log = gr.Markdown(value="")

    btn_konvertieren.click(
        fn=lambda pfad: bildformate_konvertieren(pfad) if pfad and pfad.strip() else "⚠️ Zuerst Pfad eingeben.",
        inputs=[pfad_eingabe],
        outputs=[konvert_log],
    )
    # ─── Schritt 2: Einzeldatei-Verarbeitung ───────────────
    gr.Markdown("---")
    gr.Markdown("### Schritt 2: Einzeldatei-Verarbeitung (Text + Bilder)")
    gr.Markdown(
        "Verarbeitet jede Datei einzeln: LLM liest Inhalt → benennt um → "
        "generiert Stichwörter → schreibt Metadaten in die Datei.\n\n"
        "⚠️ **Bilder:** In LM Studio muss ein Vision-Modell geladen sein, "
        "damit Bilder analysiert werden können. Textdateien funktionieren "
        "auch ohne Vision-Modell."
    )

    profil_dropdown = gr.Dropdown(
        choices=_verfuegbare_profile(),
        value=PROFIL_NAME,
        label="Profil (bestimmt Typ-Präfixe und Fachkürzel)",
        interactive=True,
    )

    with gr.Row():
        btn_verarbeiten = gr.Button(
            "🚀 Verarbeitung starten", variant="primary", scale=2)
        btn_checkpoint_reset = gr.Button(
            "🗑️ Checkpoint löschen", variant="secondary", scale=1)

    fortschritt_ausgabe = gr.Markdown(
        value="*Zuerst scannen, dann Verarbeitung starten.*")
    ergebnis_tabelle = gr.Markdown(value="")
    checkpoint_status = gr.Markdown(value="")

    # ─── Schritt 3: Übriggebliebene ───────────────────────
    gr.Markdown("---")
    gr.Markdown("### Schritt 3: Übriggebliebene Dateien markieren")
    gr.Markdown(
        "Dateien ohne erfolgreiche Verarbeitung bekommen ein Präfix:\n\n"
        "- `_binaer_` — Modelle, Archive, Binärdateien\n"
        "- `_bild_fehler_` — Bilder, die nicht erkannt wurden\n"
        "- `_text_fehler_` — Textdateien mit Extraktionsfehler\n"
        "- `_manuell_` — Kein Extraktor vorhanden"
    )

    btn_uebrige = gr.Button(
        "🔍 Übriggebliebene identifizieren", variant="primary")
    uebrige_ausgabe = gr.Markdown(
        value="*Erst Schritt 2 durchführen.*")

    gr.Markdown("Hake die Dateien an, die markiert werden sollen:")
    with gr.Row():
        btn_uebrige_alle = gr.Button(
            "☑️ Alle an/aus", variant="secondary", scale=1)
        btn_uebrige_markieren = gr.Button(
            "🏷️ Ausgewählte markieren", variant="primary", scale=2)

    uebrige_checkboxen = gr.CheckboxGroup(
        choices=[], label="Zu markierende Dateien", value=[])
    uebrige_log = gr.Markdown(value="")

    # ─── Events ────────────────────────────────────────────

    btn_scan.click(
        fn=schritt1_scannen,
        inputs=[pfad_eingabe],
        outputs=[scan_ausgabe],
    )

    btn_verarbeiten.click(
        fn=schritt2_verarbeiten,
        inputs=[profil_dropdown],
        outputs=[fortschritt_ausgabe, ergebnis_tabelle],
    )

    btn_checkpoint_reset.click(
        fn=checkpoint_zuruecksetzen,
        inputs=[],
        outputs=[checkpoint_status],
    )

    btn_uebrige.click(
        fn=schritt3_uebrige_analysieren,
        inputs=[],
        outputs=[uebrige_ausgabe, uebrige_checkboxen],
    )

    btn_uebrige_markieren.click(
        fn=schritt3_uebrige_markieren,
        inputs=[uebrige_checkboxen],
        outputs=[uebrige_log],
    )

    btn_uebrige_alle.click(
        fn=uebrige_alle_auswaehlen,
        inputs=[uebrige_checkboxen],
        outputs=[uebrige_checkboxen],
    )

app.launch(inbrowser=True)